# Model Artifact Reconstruction from Wandb

This notebook helps you download and reconstruct a TrainBundle from a saved model artifact in Weights & Biases. The artifacts were saved using the `save_train_state` function from the training loop.

## Steps:
1. Import required libraries
2. Initialize wandb and specify the artifact
3. Download the artifact
4. Reconstruct the TrainBundle from the saved state
5. Test the reconstructed model

In [1]:
import os
import tempfile
import shutil
from pathlib import Path

import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
import wandb
from flax import nnx

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Import your custom modules
from hcnn.training.train_bundle import TrainBundle

print("Libraries imported successfully!")
print(f"JAX devices: {jax.devices()}")
print(f"Working directory: {os.getcwd()}")

Libraries imported successfully!
JAX devices: [CudaDevice(id=0)]
Working directory: /home/nbehr/hcnn/notebooks
JAX devices: [CudaDevice(id=0)]
Working directory: /home/nbehr/hcnn/notebooks


In [2]:
# Initialize wandb (you can browse your artifacts at wandb.ai)
# Replace with your actual project name
PROJECT_NAME = "double_integrator_ncvx"  # Update this to match your project

# Initialize wandb without starting a new run (we just want to download artifacts)
api = wandb.Api()

print(f"Connected to wandb project: {PROJECT_NAME}")
print("Available runs in project:")

# List recent runs to help you identify the one you want
runs = api.runs(
    f"behrn-eth-zurich/{PROJECT_NAME}"
)  # Replace 'your-username' with your actual username
for i, run in enumerate(runs[:10]):  # Show last 10 runs
    print(f"  {i + 1}. Run ID: {run.id}, Name: {run.name}, State: {run.state}")
    if hasattr(run, "summary") and "loss" in run.summary:
        print(f"     Final loss: {run.summary.get('loss', 'N/A')}")
    print(f"     Created: {run.created_at}")
    print()

wandb: Currently logged in as: behrn (behrn-eth-zurich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Currently logged in as: behrn (behrn-eth-zurich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Connected to wandb project: double_integrator_ncvx
Available runs in project:
  1. Run ID: v72cj6mo, Name: vivid-dawn-5, State: crashed
     Final loss: 78.95114135742188
     Created: 2025-09-04T12:05:57Z

  2. Run ID: t5v68ptt, Name: copper-haze-6, State: crashed
     Final loss: 575.0601806640625
     Created: 2025-09-04T12:11:19Z

  3. Run ID: bm60txy4, Name: silver-violet-7, State: crashed
     Created: 2025-09-04T12:12:59Z

  4. Run ID: l7ltgl64, Name: feasible-dawn-10, State: crashed
     Created: 2025-09-04T12:16:33Z

  5. Run ID: 5d94yyez, Name: efficient-oath-11, State: crashed
     Final loss: 85.3492660522461
     Created: 2025-09-04T12:17:09Z

  6. Run ID: 6690x9xm, Name: firm-sponge-12, State: crashed
     Created: 2025-09-04T12:23:05Z

  7. Run ID: kro4dg8v, Name: fragrant-hill-13, State: crashed
     Final loss: 87.134765625
     Created: 2025-09-04T12:23:26Z

  8. Run ID: f97tgb3f, Name: smooth-pyramid-14, State: crashed
     Final loss: 93.6258773803711
     Created: 

In [3]:
# Specify the artifact you want to download
# Option 1: Specify by run ID (recommended if you know the specific run)
RUN_ID = "3955s226"  # Replace with the actual run ID from above
ARTIFACT_NAME = f"model_checkpoint_{RUN_ID}"

# Option 2: Or specify artifact directly by name and version
# ARTIFACT_NAME = "model_checkpoint_3955s226"  # Replace with actual artifact name

print(f"Looking for artifact: {ARTIFACT_NAME}")

try:
    # Download the artifact
    artifact = api.artifact(f"{PROJECT_NAME}/{ARTIFACT_NAME}:latest")

    print(f"Found artifact: {artifact.name}")
    print(f"Version: {artifact.version}")
    print(f"Size: {artifact.size}")
    print(f"Created: {artifact.created_at}")
    print(f"Description: {artifact.description}")

    # Download to a temporary directory
    artifact_dir = artifact.download()
    print(f"Downloaded artifact to: {artifact_dir}")

    # For this OCDBT checkpoint format, the checkpoint is the artifact directory itself
    # First check if there's a train_state subdirectory (old format)
    train_state_subdir = Path(artifact_dir) / "train_state"
    if train_state_subdir.exists():
        checkpoint_path = train_state_subdir
        print(f"Using subdirectory checkpoint path: {checkpoint_path}")
    else:
        # Use the artifact directory directly (OCDBT format)
        checkpoint_path = Path(artifact_dir)
        print(f"Using artifact directory as checkpoint path: {checkpoint_path}")

    print(f"Checkpoint exists: {checkpoint_path.exists()}")

    if checkpoint_path.exists():
        print("Checkpoint files:")
        for file in list(checkpoint_path.rglob("*"))[
            :10
        ]:  # Show first 10 files
            if file.is_file():
                print(f"  {file.relative_to(checkpoint_path)}")
        if len(list(checkpoint_path.rglob("*"))) > 10:
            print(
                f"  ... and {len(list(checkpoint_path.rglob('*'))) - 10} more files"
            )

except Exception as e:
    print(f"Error downloading artifact: {e}")
    print("\nTip: Make sure to:")
    print(
        "1. Replace 'your-username' with your actual wandb username in the previous cell"
    )
    print(
        "2. Replace 'your-run-id-here' with an actual run ID from the list above"
    )
    print("3. Ensure you have access to the project")
    checkpoint_path = None

Looking for artifact: model_checkpoint_3955s226
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint


Looking for artifact: model_checkpoint_3955s226
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint


wandb:   13 of 13 files downloaded.  
wandb:   13 of 13 files downloaded.  


Looking for artifact: model_checkpoint_3955s226
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint
Found artifact: model_checkpoint_3955s226:latest
Version: v0
Size: 9042918
Created: 2025-09-04T13:53:35Z
Description: HCNN training state checkpoint


wandb:   13 of 13 files downloaded.  
wandb:   13 of 13 files downloaded.  


Downloaded artifact to: /home/nbehr/hcnn/notebooks/artifacts/model_checkpoint_3955s226:v0
Using artifact directory as checkpoint path: /home/nbehr/hcnn/notebooks/artifacts/model_checkpoint_3955s226:v0
Checkpoint exists: True
Checkpoint files:
  _sharding
  _METADATA
  _CHECKPOINT_METADATA
  manifest.ocdbt
  ocdbt.process_0/manifest.ocdbt
  array_metadatas/process_0
  ... and 7 more files


In [4]:
# Load the checkpoint using Orbax
if (
    "checkpoint_path" in locals()
    and checkpoint_path
    and checkpoint_path.exists()
):
    try:
        # Create checkpointer
        checkpointer = ocp.StandardCheckpointer()

        # Load the train_state from the checkpoint path
        train_state = checkpointer.restore(checkpoint_path)

        print("Successfully loaded train_state!")
        print(f"Train state type: {type(train_state)}")
        print(
            f"Train state keys: {list(train_state.keys()) if hasattr(train_state, 'keys') else 'No keys method'}"
        )

        # Get the configuration from the run metadata
        if "RUN_ID" in locals() and RUN_ID:
            run = api.run(f"behrn-eth-zurich/{PROJECT_NAME}/{RUN_ID}")
            config = run.config

            print("\nConfiguration used for training:")
            for key, value in config.items():
                print(f"  {key}: {value}")

            print(f"\nConfig type: {type(config)}")
        else:
            print("Warning: No RUN_ID available, cannot fetch config")
            config = None

    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        import traceback

        traceback.print_exc()
        train_state = None
        config = None
else:
    print(
        "No checkpoint path available. Please run the previous cells successfully first."
    )
    train_state = None
    config = None

Successfully loaded train_state!
Train state type: <class 'dict'>
Train state keys: ['metrics', 'optimizer', 'projection_layer', 'sampler', 'state_to_rollout', 'system']

Configuration used for training:
  seed: 0
  epochs: 100
  nn_name: actions
  batch_size: 256
  system_name: double_integrator_ncvx
  solver_alpha: 1.6
  solver_sigma: 1e-06
  nn_activation: relu
  nn_hidden_dim: 512
  solver_epsilon: 1e-12
  nn_dropout_rate: 0.2
  nn_hidden_depth: 4
  solver_rho_base: 1
  solver_tol_dual: 0.1
  solver_tol_osqp: 0.001
  solver_tol_prim: 0.1
  alignment_weight: 0.1
  batches_per_epoch: 1
  rho_equality_mult: 10
  projection_horizon: 15
  solver_max_iter_lin: 500
  solver_max_iter_nonlin: 10
  optimizer_learning_rate: 0.001
  projection_relaxation_penalty: 50

Config type: <class 'dict'>

Configuration used for training:
  seed: 0
  epochs: 100
  nn_name: actions
  batch_size: 256
  system_name: double_integrator_ncvx
  solver_alpha: 1.6
  solver_sigma: 1e-06
  nn_activation: relu
  nn_

In [5]:
# Reconstruct the TrainBundle from the saved state and configuration
if train_state is not None and config is not None:
    try:
        # Convert dict config to object with attributes
        class ConfigObj:
            def __init__(self, config_dict):
                for key, value in config_dict.items():
                    setattr(self, key, value)

        config_obj = ConfigObj(config)

        # Create a new TrainBundle using the same configuration
        train_bundle = TrainBundle.factory_from_config(config_obj)

        print("Created new TrainBundle from config!")
        print(f"TrainBundle type: {type(train_bundle)}")

        # Split the bundle to get the graph definition and state
        train_graphdef, new_train_state = nnx.split(train_bundle)

        print("Split TrainBundle into graphdef and state")
        print(f"Graph def type: {type(train_graphdef)}")
        print(f"New state type: {type(new_train_state)}")

        # Replace the new state with the loaded state
        # This restores the trained parameters
        reconstructed_train_bundle = nnx.merge(train_graphdef, train_state)

        print("Successfully reconstructed TrainBundle with trained parameters!")
        print(f"Reconstructed bundle type: {type(reconstructed_train_bundle)}")

        # Verify the reconstruction by checking some properties
        print("\nTrainBundle properties:")
        print(f"  System type: {type(reconstructed_train_bundle.system)}")
        print(
            f"  State-to-rollout type: {type(reconstructed_train_bundle.state_to_rollout)}"
        )
        print(
            f"  Projection layer type: {type(reconstructed_train_bundle.projection_layer)}"
        )
        print(f"  Optimizer type: {type(reconstructed_train_bundle.optimizer)}")

        # Check if the neural network has parameters
        if hasattr(reconstructed_train_bundle.state_to_rollout, "nn"):
            print(
                f"  Neural network type: {type(reconstructed_train_bundle.state_to_rollout.nn)}"
            )
            # Try to get parameter count
            try:
                params = nnx.state(
                    reconstructed_train_bundle.state_to_rollout.nn
                )
                param_count = sum(
                    p.size
                    for p in jax.tree.leaves(params)
                    if hasattr(p, "size")
                )
                print(f"  Neural network parameter count: {param_count:,}")
            except Exception as e:
                print(f"  Could not count parameters: {e}")

    except Exception as e:
        print(f"Error reconstructing TrainBundle: {e}")
        import traceback

        traceback.print_exc()
        reconstructed_train_bundle = None
else:
    print("Cannot reconstruct TrainBundle: missing train_state or config")
    reconstructed_train_bundle = None

Created new TrainBundle from config!
TrainBundle type: <class 'hcnn.training.train_bundle.TrainBundle'>
Split TrainBundle into graphdef and state
Graph def type: <class 'flax.nnx.graph.GraphDef'>
New state type: <class 'flax.nnx.statelib.State'>
Successfully reconstructed TrainBundle with trained parameters!
Reconstructed bundle type: <class 'hcnn.training.train_bundle.TrainBundle'>

TrainBundle properties:
  System type: <class 'hcnn.training.train_bundle.System'>
  State-to-rollout type: <class 'hcnn.state_to_rollout.state_to_rollout_actions.StateToRolloutActions'>
  Projection layer type: <class 'hcnn.optimization_layer.projection_layer.ProjectionLayer'>
  Optimizer type: <class 'flax.nnx.training.optimizer.Optimizer'>


In [6]:
train_bundle.system

System(sys_dynamics=SystemDynamicsLinear(state_dim=2, action_dim=1, state_matrix_transp=Array([[1., 0.],
       [1., 1.]], dtype=float32), action_matrix_transp=Array([[0.5, 1. ]], dtype=float32)), sys_costs=SystemCostsQuadratic(state_dim=2, action_dim=1, state_cost_matrix=Array([[1., 0.],
       [0., 1.]], dtype=float32), action_cost_matrix=Array([[10.]], dtype=float32), reference_state=Array([0., 0.], dtype=float32), reference_action=Array([0.], dtype=float32), periodic_states=None), sys_constraints=SystemConstraintsMulti(state_dim=2, action_dim=1, constraint_dim=4, lower_bound=Array([-10., -10.,  -2.,   0.], dtype=float32), upper_bound=Array([10., 10.,  2., inf], dtype=float32), constraints=(SystemConstraintsBox(state_dim=2, action_dim=1, constraint_dim=3, lower_bound=Array([-10., -10.,  -2.], dtype=float32), upper_bound=Array([10., 10.,  2.], dtype=float32), active_indices=Array([0, 1, 2], dtype=int32)), SystemConstraintsBall(state_dim=2, action_dim=1, constraint_dim=1, lower_bound=